### 1. Configuração do Ambiente e Reprodutibilidade
Nesta célula, importamos as bibliotecas necessárias para manipulação de dados (`pandas`, `numpy`), visualização (`matplotlib`, `seaborn`), extração de características de texto (`sklearn`) e construção da rede neuronal (`torch`).

**Porquê as Seeds?**
Definimos `random.seed` e `torch.manual_seed` para garantir que os resultados sejam **reprodutíveis**. Sem isto, cada vez que o modelo é treinado, os pesos iniciais seriam diferentes e a performance final (Accuracy/MCC) iria variar ligeiramente em cada execução.


In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import copy # Importante para copiar o modelo na RAM

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, matthews_corrcoef, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from scipy.sparse import hstack

# Fixar seeds para reprodutibilidade
np.random.seed(42)
torch.manual_seed(42)

### 2. Funções de Apoio e Carregamento de Dados
Aqui definimos como os dados são preparados e carregados.

* **`extract_meta_features`**: Extrai métricas estilométricas (estilo de escrita) do texto original. IA e Humanos têm padrões diferentes em métricas como o tamanho médio das palavras e a diversidade do vocabulário.
* **`preprocess_text`**: Limpa o texto (conversão para minúsculas e remoção de espaços extra) para que o modelo processe o conteúdo de forma padronizada.
* **`LabelEncoder`**: Converte os nomes das categorias (Human, OpenAI, etc.) em números, pois os modelos matemáticos de Deep Learning apenas processam valores numéricos.

In [2]:
# 1. Função de Extração de Estilometria (Meta-Features)
def extract_meta_features(texts):
    features = []
    for text in texts:
        if not isinstance(text, str): text = ""
        words = text.split()
        num_words = len(words) if len(words) > 0 else 1
        
        avg_word_len = sum(len(w) for w in words) / num_words
        lexical_diversity = len(set(words)) / num_words
        uppercase_ratio = sum(1 for c in text if c.isupper()) / (len(text) if len(text) > 0 else 1)
        commas = text.count(',')
        quotes = text.count('"') + text.count("'")
        questions = text.count('?')
        
        features.append([avg_word_len, lexical_diversity, uppercase_ratio, commas, quotes, questions])
    return np.array(features)

# 2. Função de pré-processamento de texto (para o TF-IDF)
def preprocess_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    return ' '.join(text.split())

# Carregar datasets
df_treino = pd.read_csv('../data/dataset_limpo.csv', sep=';').dropna(subset=['Text', 'Label'])
df_teste = pd.read_csv('../data/dataset-exemplos.csv', sep=';').dropna(subset=['Text', 'Label'])

# Extrair Meta-Features (DO TEXTO ORIGINAL)
print("A extrair Meta-Features...")
X_train_meta = extract_meta_features(df_treino['Text'].values)
X_test_meta = extract_meta_features(df_teste['Text'].values)

# Limpar o texto para extrair TF-IDF
X_train_raw = df_treino['Text'].apply(preprocess_text).values
y_train_raw = df_treino['Label'].values

X_test_raw = df_teste['Text'].apply(preprocess_text).values
y_test_raw = df_teste['Label'].values

# Converter labels para numérico
le = LabelEncoder()
y_train_num = le.fit_transform(y_train_raw)
y_test_num = le.transform(y_test_raw)
classes = le.classes_

print(f"Classes: {classes}")
print(f"Total Treino: {len(X_train_raw)} | Total Teste: {len(X_test_raw)}")

A extrair Meta-Features...
Classes: ['Anthropic' 'Google' 'Human' 'Meta' 'OpenAI']
Total Treino: 5195 | Total Teste: 125


### 3. Vetorização de Texto (TF-IDF)
Como os computadores não entendem texto diretamente, utilizamos o **TF-IDF** para converter palavras e caracteres em matrizes numéricas que representam a importância de cada termo.

* **Word N-grams (1, 2)**: Analisa palavras isoladas e pares de palavras frequentes.
* **Char N-grams (3, 5)**: Analisa sequências de caracteres. Esta técnica é fundamental para detetar "assinaturas" mecânicas de modelos de IA (como o Antigravity) que ocorrem na estrutura das palavras e pontuação.
* **Max Features**: Limitamos a 12.000 colunas para focar o modelo nos termos mais relevantes e evitar que ele "decore" ruído estatístico.

In [3]:
print("A gerar matrizes TF-IDF (Word + Char)...")

# Word N-grams
tfidf_word = TfidfVectorizer(max_features=6000, analyzer='word', ngram_range=(1, 2), stop_words='english')
# Char N-grams
tfidf_char = TfidfVectorizer(max_features=6000, analyzer='char', ngram_range=(3, 5))

# Fit e Transform no treino
X_train_w = tfidf_word.fit_transform(X_train_raw)
X_train_c = tfidf_char.fit_transform(X_train_raw)
X_train_tfidf = hstack([X_train_w, X_train_c]).toarray() 

# Apenas Transform no teste
X_test_w = tfidf_word.transform(X_test_raw)
X_test_c = tfidf_char.transform(X_test_raw)
X_test_tfidf = hstack([X_test_w, X_test_c]).toarray()

print(f"Features extraídas! Shape: {X_train_tfidf.shape}") # Deve ser (5195, 12000)

A gerar matrizes TF-IDF (Word + Char)...
Features extraídas! Shape: (5195, 12000)


### 4. Estrutura de Dados e Arquitetura da Rede (DNN)
Preparamos a rede neuronal que irá aprender a distinguir os autores.

* **`TfidfDataset`**: Converte as matrizes em **Tensores de PyTorch**, o formato de dados necessário para o treino em GPU/CPU.
* **`LightTextDNN`**: Uma rede neuronal simplificada (256 -> 64 neurónios).
    * **Batch Normalization**: Estabiliza e acelera o treino.
    * **Dropout (0.5)**: **Elemento crucial.** Ao "desligar" 50% dos neurónios aleatoriamente, forçamos a rede a não memorizar o dataset de treino, resultando numa melhor capacidade de generalizar para textos novos (evita o *overfitting*).

In [4]:
class TfidfDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(TfidfDataset(X_train_tfidf, y_train_num), batch_size=64, shuffle=True)
test_loader = DataLoader(TfidfDataset(X_test_tfidf, y_test_num), batch_size=64, shuffle=False)

class LightTextDNN(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(LightTextDNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5), # Dropout agressivo para forçar generalização
            
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.5),
            
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        return self.net(x)

input_dim = X_train_tfidf.shape[1]
model = LightTextDNN(input_dim, len(classes))

### 5. Ciclo de Treino e Otimização Dinâmica
Nesta fase, a rede ajusta os seus pesos internos para minimizar o erro de previsão.

* **`class_weight`**: Ajusta o peso das classes para que o modelo não ignore as IAs que têm menos exemplos no dataset face aos textos humanos.
* **`AdamW`**: Algoritmo de otimização avançado que ajusta os pesos da rede.
* **`ReduceLROnPlateau`**: Reduz a taxa de aprendizagem se o modelo parar de evoluir, permitindo um ajuste mais fino dos pesos.
* **Checkpoint**: O código guarda apenas o estado do modelo que atingiu o melhor **MCC** durante as épocas.

In [5]:
# Reiniciar pesos das classes e optimizer para o novo modelo
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_num), y=y_train_num)
weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

best_mcc = -1
patience_counter = 0

print("A iniciar o treino da rede simplificada...")
for epoch in range(60): # 60 épocas costumam chegar para esta arquitetura
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
    
    model.eval()
    test_preds, test_targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            test_preds.extend(preds.numpy())
            test_targets.extend(y_batch.numpy())
    
    test_mcc = matthews_corrcoef(test_targets, test_preds)
    scheduler.step(test_mcc)
    
    if test_mcc > best_mcc:
        best_mcc = test_mcc
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model_light.pt')
    else:
        patience_counter += 1
        if patience_counter >= 10: break

    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}] | Test MCC: {test_mcc:.4f} | Best MCC: {best_mcc:.4f}")

model.load_state_dict(torch.load('best_model_light.pt'))
print(f"Treino concluído! Melhor MCC no dataset do professor: {best_mcc:.4f}")

A iniciar o treino da rede simplificada...
Epoch [5] | Test MCC: 0.5531 | Best MCC: 0.5952
Epoch [10] | Test MCC: 0.6047 | Best MCC: 0.6600
Epoch [15] | Test MCC: 0.6380 | Best MCC: 0.6600
Treino concluído! Melhor MCC no dataset do professor: 0.6600


### 6. Avaliação de Resultados (Dataset do Professor)
Validamos a eficácia real do modelo utilizando o dataset de exemplos fornecido, que o modelo nunca viu durante o treino.

* **Matthews Correlation Coefficient (MCC)**: É a nossa métrica de sucesso. É superior à Accuracy em problemas multiaclasse, pois avalia a qualidade da classificação de forma equilibrada para todas as categorias.
* **Relatório de Classificação**: Detalha a precisão (falsos positivos) e o recall (capacidade de encontrar a classe) para cada autor.

In [6]:
# Carregar o melhor modelo da rede
model.load_state_dict(torch.load('best_model_light.pt'))
model.eval()

y_pred_num = []
y_targets_num = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        _, preds = torch.max(outputs, 1)
        y_pred_num.extend(preds.numpy())
        y_targets_num.extend(y_batch.numpy())

y_pred_labels = le.inverse_transform(y_pred_num)
y_test_labels = le.inverse_transform(y_targets_num)

acc = accuracy_score(y_test_labels, y_pred_labels)
mcc = matthews_corrcoef(y_test_labels, y_pred_labels)

print(f"Accuracy Rede: {acc*100:.2f}%")
print(f"MCC Rede: {mcc:.4f}\n")

print("Relatório de Classificação:")
print(classification_report(y_test_labels, y_pred_labels, target_names=classes))

Accuracy Rede: 75.20%
MCC Rede: 0.6600

Relatório de Classificação:
              precision    recall  f1-score   support

   Anthropic       0.79      0.65      0.71        23
      Google       0.56      0.62      0.59        16
       Human       0.81      0.96      0.88        52
        Meta       0.71      0.59      0.65        17
      OpenAI       0.75      0.53      0.62        17

    accuracy                           0.75       125
   macro avg       0.72      0.67      0.69       125
weighted avg       0.75      0.75      0.74       125



### 7. Exportação da Submissão Final
Aplicamos o modelo final treinado ao ficheiro de submissão cego (`subm3.csv`). 

Garantimos que o pré-processamento e a vetorização (TF-IDF) seguem exatamente os mesmos passos do treino. O resultado é um ficheiro CSV com as previsões finais, pronto para submissão na plataforma.

In [7]:
caminho_submissao = '../subm3.csv' 

if os.path.exists(caminho_submissao):
    df_subm = pd.read_csv(caminho_submissao, sep=';')
    print("A gerar Submissão 3...")
    
    # 1. Limpar o texto e transformar APENAS em TF-IDF
    X_subm_raw = df_subm['Text'].apply(preprocess_text).values
    X_subm_w = tfidf_word.transform(X_subm_raw)
    X_subm_c = tfidf_char.transform(X_subm_raw)
    X_subm_tfidf = hstack([X_subm_w, X_subm_c]).toarray()
    
    # 2. Converter para Tensor
    X_subm_tensor = torch.tensor(X_subm_tfidf, dtype=torch.float32)
    
    # 3. Inferência
    model.eval()
    with torch.no_grad():
        subm_outputs = model(X_subm_tensor)
        _, subm_predicted = torch.max(subm_outputs, 1)
        
    # 4. Inverter labels
    df_subm['Label'] = le.inverse_transform(subm_predicted.numpy())
    
    output_dir = '../Subm3'
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, 'subm3-g3-MIA-A.csv')
    df_subm[['ID', 'Label']].to_csv(output_path, sep=';', index=False)
    
    print(f"Ficheiro de Submissão gerado em: {output_path}")
    print("\nDistribuição das Previsões:")
    print(df_subm['Label'].value_counts())
else:
    print(f"Ficheiro {caminho_submissao} não encontrado.")

A gerar Submissão 3...
Ficheiro de Submissão gerado em: ../Subm3/subm3-g3-MIA-A.csv

Distribuição das Previsões:
Label
Human        58
Meta         26
OpenAI       25
Anthropic    22
Google       19
Name: count, dtype: int64
